## Implementing Async Tool Execution

# Unit 25: Thread Pooling, Cooperative Task Scheduling, and Concurrent Fan-Out

## Parallelizing Tool Execution Matrices

Welcome back! In the previous lesson, you re-engineered your agent architecture to use non-blocking asynchronous programming, enabling a single agent coordinator instance to drive multiple independent user conversations simultaneously.

However, we also identified a major architectural bottleneck: **tool execution still happens sequentially**. Whenever any concurrent conversation triggers a local backend function, the core event loop thread blocks, forcing all adjacent agent streams into a holding pattern.

In this lesson, we will eliminate this performance bottleneck by parallelizing our tool execution layer. You will learn how to transition legacy synchronous tool functions into non-blocking background workers and use concurrent scheduling to fire multiple tools simultaneously across different conversation lifecycles.

---

## Understanding the Tool Execution Bottleneck

To build ultra-low-latency agent meshes, we must understand how blocking synchronous operations can hurt an event loop. Declaring `async def run()` allows your system to step away from a thread while waiting for remote HTTP tokens from Claude's servers. However, when Claude responds with a `tool_use` directive, our previous implementation fell back onto a standard, sequential `for` loop.

Consider what happens under a sequential execution model when multiple active agent turns resolve over the wire at the same time:

```text
SEQUENTIAL BLOCKING TOOL PATH:
┌────────────────────────────────────────────────────────────────────────┐
│ Turn 1 (Math Thread): Run sum_numbers()  ◄── Threads block globally    │
└────────────────────────────────────────────────────────────────────────┘
                                          ▼
┌────────────────────────────────────────────────────────────────────────┐
│ Turn 2 (Risk Thread): Run check_db()     ◄── Waited for Turn 1 to close│
└────────────────────────────────────────────────────────────────────────┘

```

This structural architecture creates an execution queue. If an agent turn utilizes a tool that interacts with an external service (like fetching a database write or hitting a third-party weather API) that takes **2 seconds** to close its socket, three concurrent conversations running that tool will take **6 seconds** to complete.

Even though your upstream connections to Claude are async, the synchronous nature of the local functions blocks the event loop thread, completely freezing your application during network round-trips. To resolve this, we must make our tool execution path asynchronous, enabling multiple backend functions to run concurrently.

---

## Running Synchronous Functions Asynchronously via `asyncio.to_thread`

Your core tool libraries—such as `sum_numbers()` or `multiply_numbers()`—are typically written as regular, blocking synchronous functions. While you could rewrite your entire codebase as native async functions using `async def`, doing so can be impractical when dealing with massive legacy repositories or third-party dependencies.

Fortunately, Python's `asyncio` module provides an elegant solution called `asyncio.to_thread()`. This utility takes a standard synchronous function, offloads it to a separate worker thread inside an isolated background pool, and returns a non-blocking coroutine promise. This prevents the function from freezing the main event loop.

Let's update the orchestrator's `call_tool()` method to use this thread-pooling pattern:

```python
import asyncio

async def call_tool(self, tool_use: getattr) -> dict:
    """
    Asynchronously executes a synchronous tool function within an isolated thread pool.
    """
    tool_name = tool_use.name
    tool_input = tool_use.input or {}
    tool_use_id = tool_use.id
    
    print(f"🔧 Offloading Tool Call: {tool_name}({tool_input})")
    
    try:
        tool_fn = self.tools[tool_name]
    except KeyError:
        result = f"Error: Tool '{tool_name}' not found in local registry."
    except Exception as e:
        result = f"Error during lookup: {str(e)}"
    else:
        try:
            # Offload the blocking execution to a separate thread pool automatically
            result = str(await asyncio.to_thread(tool_fn, **tool_input))
        except Exception as e:
            result = f"Runtime Exception inside '{tool_name}': {str(e)}"
            
    return {
        "type": "tool_result",
        "tool_use_id": tool_use_id,
        "content": result
    }

```

### Deconstructing the Thread Abstraction:

* **`async def call_tool(...)`:** Converting the method definition to an async coroutine is required because we are introducing an internal `await` expression.
* **`await asyncio.to_thread(tool_fn, tool_input)`:** Instead of calling the function directly on the main thread, this wraps the execution. The event loop hands off the blocking operation to a background thread pool and immediately switches context to process other pending tasks.

---

## Collecting Tool Calls for Concurrent Execution

Now that our individual tool executor is non-blocking, we must adjust the core `run()` loop to handle concurrent fan-out. Instead of immediately awaiting each tool call one by one during iteration, we will wrap them into an array of deferred task promises.

Update your agent's turn evaluation loop inside `agent.py`:

```python
if response.stop_reason == "tool_use":
    tool_results = []
    
    # Instantiate a collection array to hold scheduled task futures
    tasks = []
    
    for content_item in response.content:
        if content_item.type == "tool_use":
            # Intercept and process multi-agent routing handoffs immediately
            if content_item.name == "handoff":
                handoff_success, handoff_result = await self.call_handoff(content_item, messages)
                if handoff_success:
                    return handoff_result
                else:
                    tool_results.append(handoff_result)
            else:
                # Schedule standard tools as concurrent background tasks
                task_future = asyncio.create_task(self.call_tool(content_item))
                tasks.append(task_future)

```

### Understanding Task Scheduling:

By wrapping our method in `asyncio.create_task()`, we instruct Python's event loop to schedule the coroutine for immediate execution in the background without halting the current loop iteration. This function returns a `Task` object wrapper that can be awaited later, allowing us to build a list of active operations that execute simultaneously.

---

## Executing All Scheduled Tools in Parallel via `asyncio.gather`

Once the turn loop finishes scanning Claude's response payload and populates the `tasks` array, we can execute the scheduled actions concurrently using `asyncio.gather()`.

```python
    # Ensure the task collection list contains elements before triggering gather
    if tasks:
        # Fire all background thread pool tasks simultaneously across the event loop
        resolved_tool_blocks = await asyncio.gather(*tasks)
        
        # Extend our main results tracking array with the resolved payloads
        tool_results.extend(resolved_tool_blocks)
        
    # Append the combined results to the conversation history
    messages.append({
        "role": "user",
        "content": tool_results
    })

```

### The Unpacking Operator (`*`):

The asterisk (`*`) symbol unpacks your python list container, presenting every entry within the array as an independent argument variable to `asyncio.gather()`. The gather method manages the execution of all tasks simultaneously, pauses until the slowest background thread returns, and then aggregates the completed data points back into a clean list.

---

## Complete Multi-Agent Parallel Loop Blueprint

Let's look at the consolidated architecture for our agentic task execution block. This complete structure manages the balance between immediate routing handoffs and concurrent tool execution:

```python
if response.stop_reason == "tool_use":
    tool_results = []
    tasks = []
    
    # Phase 1: Scan, Filter, and Instantly Schedule Background Tasks
    for content_item in response.content:
        if content_item.type == "tool_use":
            if content_item.name == "handoff":
                # Multi-agent routing handoffs are handled immediately
                handoff_success, handoff_result = await self.call_handoff(content_item, messages)
                if handoff_success:
                    return handoff_result
                else:
                    tool_results.append(handoff_result)
            else:
                # Schedule regular calculations or lookups for concurrent fan-out
                tasks.append(asyncio.create_task(self.call_tool(content_item)))
                
    # Phase 2: Fire Asynchronous Concurrent Fan-out Pass
    if tasks:
        resolved_outputs = await asyncio.gather(*tasks)
        tool_results.extend(resolved_outputs)
        
    # Phase 3: Update Conversation History and Resubmit State
    messages.append({
        "role": "user",
        "content": tool_results
    })

```

---

## Observing Non-Blocking Concurrency in Action

When you run your updated multi-conversation orchestrator, the terminal logs show a highly optimized interleaving pattern across your concurrent threads:

```text
Starting 2 research operations in parallel matrix...

🔧 Tool called: sum_numbers({'a': 2, 'b': 3})
🔧 Tool called: power({'base': -5, 'exponent': 2})
🔧 Tool called: multiply_numbers({'a': 4, 'b': 4})
🔧 Tool called: multiply_numbers({'a': 4, 'b': 1})
...

=== run 1 ===
Final response:
The answer is **80**.

=== run 2 ===
Final response:
The roots of the equation x² - 5x + 6 = 0 are x = 3 and x = 2.

```

### Performance Trace Analysis:

Notice how the tool tracking logs from separate conversation blocks burst onto the screen **simultaneously**. Run 1 handles an arithmetic expression while Run 2 processes a quadratic equation root finder concurrently.

Instead of waiting for Run 1 to close all its tool steps sequentially, both tasks make progress at the same time. This drops your total execution latency down to match the processing window of your single longest running task ($T \approx \max(t_n)$) rather than stacking them linearly, maximizing the performance and efficiency of your agent system.

---

## Summary & Practice Exercises

You have successfully removed the tool execution bottleneck from your autonomous architectures. You learned how to use `asyncio.to_thread()` to run blocking legacy functions inside thread pools without freezing the main process, how to compile background tasks using `asyncio.create_task()`, and how to handle concurrent fan-out using `asyncio.gather()`.

Let's head into our practice code labs to implement this concurrent execution pattern across diverse multi-agent scenarios!

## Making Tool Execution Async

Now it's time to transform your tool execution from blocking to non-blocking by making the call_tool() method async.

In the previous lesson, you learned how asyncio.to_thread() allows synchronous functions to run without blocking the event loop, and now you'll apply that knowledge to enable concurrent tool execution in your agent system.

Your task is to update the call_tool() method in src/agent.py to use async programming. You'll need to:

    Import the asyncio module at the top of the file.
    Add the async keyword to the call_tool() method definition.
    Replace the direct synchronous tool function call with asyncio.to_thread() and await.
    Update the run() method to await the async call_tool() method.

By making call_tool() async and using asyncio.to_thread(), each tool runs in a background thread without blocking the event loop, so multiple conversations can progress concurrently. Within a single turn tools still run one-by-one; in the next exercise you'll batch them with create_task() and gather() for true parallel execution.

```
# agent.py
# TODO: Import asyncio module to enable async tool execution
from anthropic import AsyncAnthropic


class Agent:
    BASE_SYSTEM_PROMPT = (
        "You are an autonomous agent that can take multiple tool-calling steps when helpful. "
        "The user only sees your response when you stop using tools, not your tool usage or reasoning steps. "
        "When you provide your answer without calling tools, make it complete and standalone.\n"
        "Additional instructions:\n"
    )

    def __init__(
        self,
        name,
        system_prompt="You are a helpful assistant.",
        model="claude-sonnet-4-6",
        tools=None,
        tool_schemas=None,
        handoffs=None,
        max_turns=10
    ):
        self.client = AsyncAnthropic()
        self.name = name
        self.model = model
        self.system_prompt = self.BASE_SYSTEM_PROMPT + system_prompt
        self.max_turns = max_turns

        self.tools = {} if tools is None else dict(tools)
        self.tool_schemas = [] if tool_schemas is None else list(tool_schemas)
        self.handoffs = [] if handoffs is None else list(handoffs)
        
        self.handoff_schema = {
            "name": "handoff",
            "description": "Transfer control to another specialized agent. Use this when the user's request is better handled by a different agent.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": f"Name of the agent to handoff to. Available agents: {[agent.name for agent in self.handoffs]}"
                    },
                    "reason": {
                        "type": "string", 
                        "description": "Brief explanation of why this handoff is needed"
                    }
                },
                "required": ["name", "reason"]
            }
        }
        
    def _extract_text(self, content):
        return "".join(
            block.text for block in content
            if getattr(block, "type", None) == "text"
        )    

    def _build_request_args(self, messages):
        request_args = {
            "model": self.model,
            "system": self.system_prompt,
            "messages": messages,
            "max_tokens": 8000,
        }

        all_tools = []
        
        if self.tool_schemas:
            all_tools.extend(self.tool_schemas)
        if self.handoffs:
            all_tools.append(self.handoff_schema)
        if all_tools:
            request_args["tools"] = all_tools

        return request_args

    # TODO: Make this method async by adding the async keyword
    def call_tool(self, tool_use):
        tool_name = tool_use.name
        tool_input = tool_use.input or {}
        tool_use_id = tool_use.id

        print(f"🔧 Tool called: {tool_name}({tool_input})")
        
        try:
            tool_fn = self.tools[tool_name]
        except KeyError:
            result = f"Error: Tool {tool_name} not found"
        except Exception as e:
            result = f"Error: {str(e)}"
        else:
            try:
                # TODO: Use asyncio.to_thread() with await to run the tool function without blocking
                result = str(tool_fn(**tool_input))
            except Exception as e:
                result = f"Error: {str(e)}"

        return {
            "type": "tool_result",
            "tool_use_id": tool_use_id,
            "content": result
        }
        
    async def call_handoff(self, tool_use, messages):
        agent_name = tool_use.input.get("name")
        reason = tool_use.input.get("reason", "No reason provided")
        
        print(f"🔄 Handoff to: {agent_name}")
        print(f"📝 Reason: {reason}")
        
        try:
            target_agent = next(agent for agent in self.handoffs if agent.name == agent_name)
            clean_messages = messages[:-1] if messages and messages[-1]["role"] == "assistant" else messages
            result = await target_agent.run(clean_messages)

            return True, result
        except StopIteration:
            return False, {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": f"Handoff failed: Agent '{agent_name}' not found. Available agents: {[agent.name for agent in self.handoffs]}"
            }
        except Exception as e:
            return False, {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": f"Handoff failed: Error during handoff to '{agent_name}': {str(e)}"
            }

    async def run(self, input_messages):
        messages = input_messages.copy()

        turn = 0
        while turn < self.max_turns:
            turn += 1

            response = await self.client.messages.create(
                **self._build_request_args(messages)
            )

            messages.append({"role": "assistant", "content": response.content})

            if response.stop_reason == "tool_use":
                tool_results = []
                for content_item in response.content:
                    if content_item.type == "tool_use":
                        if content_item.name == "handoff":
                            handoff_success, handoff_result = await self.call_handoff(content_item, messages)
                            if handoff_success:
                                return handoff_result
                            else:
                                tool_results.append(handoff_result)
                        else:
                            # TODO: Add await keyword to call the async call_tool method
                            tool_result = self.call_tool(content_item)
                            tool_results.append(tool_result)

                messages.append({
                    "role": "user",
                    "content": tool_results
                })
        
            else:
                response_text = self._extract_text(response.content)

                return messages, response_text

        # If the max turns is reached, raise an exception
        raise Exception("Max turns reached")

# main.py
import asyncio
import json
from agent import Agent
from functions import (
    sum_numbers,
    multiply_numbers,
    subtract_numbers,
    divide_numbers,
    power,
    square_root
)

# Load the schemas from JSON file
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Create a dictionary mapping tool names to functions
tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers,
    "subtract_numbers": subtract_numbers,
    "divide_numbers": divide_numbers,
    "power": power,
    "square_root": square_root
}

async def main():
    # Write multiple prompts that we'll process concurrently
    prompts = [
        "Solve this: (2 + 3) * (4*4)",
        "Find the roots of x^2 - 5x + 6 = 0",
    ]

    # Create a single agent that will handle multiple conversations concurrently
    agent = Agent(
        name="math_assistant",
        system_prompt="You are a helpful math assistant.",
        tools=tools,
        tool_schemas=tool_schemas,
        max_turns=15
    )

    # Create a list of coroutine tasks, one for each prompt (not started yet)
    tasks = [
        agent.run([{"role": "user", "content": prompt}])
        for prompt in prompts
    ]

    # Run all tasks concurrently and wait for all to complete
    results = await asyncio.gather(*tasks)

    # Display the final response from each conversation
    for idx, (_, result) in enumerate(results, start=1):
        print(f"\n=== run {idx} ===")
        print(result)

# Entry point - creates event loop and runs the async main() function
if __name__ == "__main__":
    asyncio.run(main())


```

To eliminate the execution blocking bottleneck, we need to transform our local tool execution module into a non-blocking coroutine.

By utilizing `asyncio.to_thread()`, we can offload synchronous functions onto an independent internal background thread pool, freeing Python's main event loop to immediately coordinate adjacent tasks while the function runs.

Here is the complete implementation of `agent.py` updated with async tool invocation hooks, followed by the verification file `main.py`:

### 1. Non-Blocking Tool Orchestrator (`agent.py`)

```python
# Import asyncio module to enable async tool execution
import asyncio
from anthropic import AsyncAnthropic


class Agent:
    BASE_SYSTEM_PROMPT = (
        "You are an autonomous agent that can take multiple tool-calling steps when helpful. "
        "The user only sees your response when you stop using tools, not your tool usage or reasoning steps. "
        "When you provide your answer without calling tools, make it complete and standalone.\n"
        "Additional instructions:\n"
    )

    def __init__(
        self,
        name,
        system_prompt="You are a helpful assistant.",
        model="claude-sonnet-4-6",
        tools=None,
        tool_schemas=None,
        handoffs=None,
        max_turns=10
    ):
        self.client = AsyncAnthropic()
        self.name = name
        self.model = model
        self.system_prompt = self.BASE_SYSTEM_PROMPT + system_prompt
        self.max_turns = max_turns

        self.tools = {} if tools is None else dict(tools)
        self.tool_schemas = [] if tool_schemas is None else list(tool_schemas)
        self.handoffs = [] if handoffs is None else list(handoffs)
        
        self.handoff_schema = {
            "name": "handoff",
            "description": "Transfer control to another specialized agent. Use this when the user's request is better handled by a different agent.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": f"Name of the agent to handoff to. Available agents: {[agent.name for agent in self.handoffs]}"
                    },
                    "reason": {
                        "type": "string", 
                        "description": "Brief explanation of why this handoff is needed"
                    }
                },
                "required": ["name", "reason"]
            }
        }
        
    def _extract_text(self, content):
        return "".join(
            block.text for block in content
            if getattr(block, "type", None) == "text"
        )    

    def _build_request_args(self, messages):
        request_args = {
            "model": self.model,
            "system": self.system_prompt,
            "messages": messages,
            "max_tokens": 8000,
        }

        all_tools = []
        
        if self.tool_schemas:
            all_tools.extend(self.tool_schemas)
        if self.handoffs:
            all_tools.append(self.handoff_schema)
        if all_tools:
            request_args["tools"] = all_tools

        return request_args

    # Made this method async by adding the async keyword
    async def call_tool(self, tool_use):
        tool_name = tool_use.name
        tool_input = tool_use.input or {}
        tool_use_id = tool_use.id

        print(f"🔧 Tool called: {tool_name}({tool_input})")
        
        try:
            tool_fn = self.tools[tool_name]
        except KeyError:
            result = f"Error: Tool {tool_name} not found"
        except Exception as e:
            result = f"Error: {str(e)}"
        else:
            try:
                # Used asyncio.to_thread() with await to run the tool function without blocking
                result = str(await asyncio.to_thread(tool_fn, **tool_input))
            except Exception as e:
                result = f"Error: {str(e)}"

        return {
            "type": "tool_result",
            "tool_use_id": tool_use_id,
            "content": result
        }
        
    async def call_handoff(self, tool_use, messages):
        agent_name = tool_use.input.get("name")
        reason = tool_use.input.get("reason", "No reason provided")
        
        print(f"🔄 Handoff to: {agent_name}")
        print(f"📝 Reason: {reason}")
        
        try:
            target_agent = next(agent for agent in self.handoffs if agent.name == agent_name)
            clean_messages = messages[:-1] if messages and messages[-1]["role"] == "assistant" else messages
            result = await target_agent.run(clean_messages)

            return True, result
        except StopIteration:
            return False, {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": f"Handoff failed: Agent '{agent_name}' not found. Available agents: {[agent.name for agent in self.handoffs]}"
            }
        except Exception as e:
            return False, {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": f"Handoff failed: Error during handoff to '{agent_name}': {str(e)}"
            }

    async def run(self, input_messages):
        messages = input_messages.copy()

        turn = 0
        while turn < self.max_turns:
            turn += 1

            response = await self.client.messages.create(
                **self._build_request_args(messages)
            )

            messages.append({"role": "assistant", "content": response.content})

            if response.stop_reason == "tool_use":
                tool_results = []
                for content_item in response.content:
                    if content_item.type == "tool_use":
                        if content_item.name == "handoff":
                            handoff_success, handoff_result = await self.call_handoff(content_item, messages)
                            if handoff_success:
                                return handoff_result
                            else:
                                tool_results.append(handoff_result)
                        else:
                            # Added await keyword to call the async call_tool method
                            tool_result = await self.call_tool(content_item)
                            tool_results.append(tool_result)

                messages.append({
                    "role": "user",
                    "content": tool_results
                })
        
            else:
                response_text = self._extract_text(response.content)

                return messages, response_text

        raise Exception("Max turns reached")

```

---

### 2. Verification Runtime Orchestration (`main.py`)

```python
# main.py
import asyncio
import json
from agent import Agent
from functions import (
    sum_numbers,
    multiply_numbers,
    subtract_numbers,
    divide_numbers,
    power,
    square_root
)

# Load the schemas from JSON file
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Create a dictionary mapping tool names to functions
tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers,
    "subtract_numbers": subtract_numbers,
    "divide_numbers": divide_numbers,
    "power": power,
    "square_root": square_root
}

async def main():
    # Write multiple prompts that we'll process concurrently
    prompts = [
        "Solve this: (2 + 3) * (4*4)",
        "Find the roots of x^2 - 5x + 6 = 0",
    ]

    # Create a single agent that will handle multiple conversations concurrently
    agent = Agent(
        name="math_assistant",
        system_prompt="You are a helpful math assistant.",
        tools=tools,
        tool_schemas=tool_schemas,
        max_turns=15
    )

    # Create a list of coroutine tasks, one for each prompt (not started yet)
    tasks = [
        agent.run([{"role": "user", "content": prompt}])
        for prompt in prompts
    ]

    # Run all tasks concurrently and wait for all to complete
    results = await asyncio.gather(*tasks)

    # Display the final response from each conversation
    for idx, (_, result) in enumerate(results, start=1):
        print(f"\n=== run {idx} ===")
        print(result)

# Entry point - creates event loop and runs the async main() function
if __name__ == "__main__":
    asyncio.run(main())

```

### Key Concurrency Shifts Added:

* **Background Offloading Strategy:** By using `await asyncio.to_thread(tool_fn, tool_input)`, the primary event loop thread stays free. If another concurrent conversation stream needs to calculate expressions or make network calls, the engine switches context instantly.
* **Turn Sequence Consistency:** Adding `await self.call_tool(content_item)` keeps the sequential logic intact *within a single conversation turn* while allowing *independent conversations* to run without blocking each other. In your next exercise, you'll optimize this pattern to support parallel batching within the same turn!

## Testing Parallel Tool Execution

Excellent work! In the previous exercise, you made the call_tool() method async using asyncio.to_thread(). Now, before integrating parallel tool execution into the full agent workflow, you'll experiment with it directly to understand how asyncio.create_task() and asyncio.gather() work together.

Your task is to complete the test function in src/main.py to execute multiple tool calls in parallel:

    Create a list of tasks using asyncio.create_task() for each of the three mock tool_use objects.
    Use asyncio.gather() with await to execute all tasks concurrently and collect their results.
    Print the results with a descriptive header and loop through each result.

The key insight is that asyncio.create_task() schedules a coroutine to run without waiting for it, and asyncio.gather() executes multiple tasks concurrently. By calling the async call_tool() method directly with this pattern, you'll see firsthand how multiple tools can execute simultaneously. This hands-on experience will prepare you to integrate the same pattern into the agent's run() method in the next exercise!

```
# main.py
import asyncio
import json
from agent import Agent
from functions import sum_numbers, multiply_numbers


# Simple class to mock a tool_use object from Claude's API
class MockToolUse:
    def __init__(self, name, input_data, id):
        self.name = name
        self.input = input_data
        self.id = id


# Load the schemas from JSON file
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Create a dictionary mapping tool names to functions
tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers,
}

# Create an agent with tools
agent = Agent(
    name="math_assistant",
    system_prompt="You are a helpful math assistant.",
    tools=tools,
    tool_schemas=tool_schemas,
)


async def test_async_tool():
    # Create multiple mock tool_use objects
    mock_tool_use_1 = MockToolUse(
        name="sum_numbers",
        input_data={"a": 5, "b": 3},
        id="test_123"
    )
    
    mock_tool_use_2 = MockToolUse(
        name="multiply_numbers",
        input_data={"a": 4, "b": 7},
        id="test_456"
    )
    
    mock_tool_use_3 = MockToolUse(
        name="sum_numbers",
        input_data={"a": 10, "b": 20},
        id="test_789"
    )

    # TODO: Create a list of tasks using asyncio.create_task() for each mock tool_use object
    # Schedule agent.call_tool() for mock_tool_use_1, mock_tool_use_2, and mock_tool_use_3

    # TODO: Use asyncio.gather() with await to execute all tasks concurrently and store results

    # TODO: Print the results with a descriptive header and loop through each result


if __name__ == "__main__":
    asyncio.run(test_async_tool())


# agent.py
import asyncio
from anthropic import AsyncAnthropic


class Agent:
    BASE_SYSTEM_PROMPT = (
        "You are an autonomous agent that can take multiple tool-calling steps when helpful. "
        "The user only sees your response when you stop using tools, not your tool usage or reasoning steps. "
        "When you provide your answer without calling tools, make it complete and standalone.\n"
        "Additional instructions:\n"
    )

    def __init__(
        self,
        name,
        system_prompt="You are a helpful assistant.",
        model="claude-sonnet-4-6",
        tools=None,
        tool_schemas=None,
        handoffs=None,
        max_turns=10
    ):
        self.client = AsyncAnthropic()
        self.name = name
        self.model = model
        self.system_prompt = self.BASE_SYSTEM_PROMPT + system_prompt
        self.max_turns = max_turns

        self.tools = {} if tools is None else dict(tools)
        self.tool_schemas = [] if tool_schemas is None else list(tool_schemas)
        self.handoffs = [] if handoffs is None else list(handoffs)
        
        self.handoff_schema = {
            "name": "handoff",
            "description": "Transfer control to another specialized agent. Use this when the user's request is better handled by a different agent.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": f"Name of the agent to handoff to. Available agents: {[agent.name for agent in self.handoffs]}"
                    },
                    "reason": {
                        "type": "string", 
                        "description": "Brief explanation of why this handoff is needed"
                    }
                },
                "required": ["name", "reason"]
            }
        }
        
    def _extract_text(self, content):
        return "".join(
            block.text for block in content
            if getattr(block, "type", None) == "text"
        )    

    def _build_request_args(self, messages):
        request_args = {
            "model": self.model,
            "system": self.system_prompt,
            "messages": messages,
            "max_tokens": 8000,
        }

        all_tools = []
        
        if self.tool_schemas:
            all_tools.extend(self.tool_schemas)
        if self.handoffs:
            all_tools.append(self.handoff_schema)
        if all_tools:
            request_args["tools"] = all_tools

        return request_args

    async def call_tool(self, tool_use):
        tool_name = tool_use.name
        tool_input = tool_use.input or {}
        tool_use_id = tool_use.id

        print(f"🔧 Tool called: {tool_name}({tool_input})")
        
        try:
            tool_fn = self.tools[tool_name]
        except KeyError:
            result = f"Error: Tool {tool_name} not found"
        except Exception as e:
            result = f"Error: {str(e)}"
        else:
            try:
                result = str(await asyncio.to_thread(tool_fn, **tool_input))
            except Exception as e:
                result = f"Error: {str(e)}"

        return {
            "type": "tool_result",
            "tool_use_id": tool_use_id,
            "content": result
        }
        
    async def call_handoff(self, tool_use, messages):
        agent_name = tool_use.input.get("name")
        reason = tool_use.input.get("reason", "No reason provided")
        
        print(f"🔄 Handoff to: {agent_name}")
        print(f"📝 Reason: {reason}")
        
        try:
            target_agent = next(agent for agent in self.handoffs if agent.name == agent_name)
            clean_messages = messages[:-1] if messages and messages[-1]["role"] == "assistant" else messages
            result = await target_agent.run(clean_messages)

            return True, result
        except StopIteration:
            return False, {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": f"Handoff failed: Agent '{agent_name}' not found. Available agents: {[agent.name for agent in self.handoffs]}"
            }
        except Exception as e:
            return False, {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": f"Handoff failed: Error during handoff to '{agent_name}': {str(e)}"
            }

    async def run(self, input_messages):
        messages = input_messages.copy()

        turn = 0
        while turn < self.max_turns:
            turn += 1

            response = await self.client.messages.create(
                **self._build_request_args(messages)
            )

            messages.append({"role": "assistant", "content": response.content})

            if response.stop_reason == "tool_use":
                tool_results = []
                for content_item in response.content:
                    if content_item.type == "tool_use":
                        if content_item.name == "handoff":
                            handoff_success, handoff_result = await self.call_handoff(content_item, messages)
                            if handoff_success:
                                return handoff_result
                            else:
                                tool_results.append(handoff_result)
                        else:
                            tool_result = await self.call_tool(content_item)
                            tool_results.append(tool_result)

                messages.append({
                    "role": "user",
                    "content": tool_results
                })
        
            else:
                response_text = self._extract_text(response.content)

                return messages, response_text

        # If the max turns is reached, raise an exception
        raise Exception("Max turns reached")


```

To execute multiple tool functions concurrently, we combine `asyncio.create_task()` and `asyncio.gather()`.

Calling `asyncio.create_task(agent.call_tool(...))` constructs an active task on the event loop immediately, which begins executing the background thread offloading. We collect these references in an array and unpack them inside `asyncio.gather()` to wait for all parallel threads to resolve.

Here is the complete, working implementation for your test layout inside `main.py`:

```python
# main.py
import asyncio
import json
from agent import Agent
from functions import sum_numbers, multiply_numbers


# Simple class to mock a tool_use object from Claude's API
class MockToolUse:
    def __init__(self, name, input_data, id):
        self.name = name
        self.input = input_data
        self.id = id


# Load the schemas from JSON file
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Create a dictionary mapping tool names to functions
tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers,
}

# Create an agent with tools
agent = Agent(
    name="math_assistant",
    system_prompt="You are a helpful math assistant.",
    tools=tools,
    tool_schemas=tool_schemas,
)


async def test_async_tool():
    # Create multiple mock tool_use objects
    mock_tool_use_1 = MockToolUse(
        name="sum_numbers",
        input_data={"a": 5, "b": 3},
        id="test_123"
    )
    
    mock_tool_use_2 = MockToolUse(
        name="multiply_numbers",
        input_data={"a": 4, "b": 7},
        id="test_456"
    )
    
    mock_tool_use_3 = MockToolUse(
        name="sum_numbers",
        input_data={"a": 10, "b": 20},
        id="test_789"
    )

    # Create a list of tasks using asyncio.create_task() for each mock tool_use object
    # This schedules the tasks on the event loop concurrently without waiting yet
    tasks = [
        asyncio.create_task(agent.call_tool(mock_tool_use_1)),
        asyncio.create_task(agent.call_tool(mock_tool_use_2)),
        asyncio.create_task(agent.call_tool(mock_tool_use_3))
    ]

    # Use asyncio.gather() with await to execute all tasks concurrently and store results
    print("⏳ Fanning out tool calls concurrently via asyncio.gather...")
    results = await asyncio.gather(*tasks)

    # Print the results with a descriptive header and loop through each result
    print("\n📊 === PARALLEL TOOL EXECUTION RESULTS ===")
    for idx, result in enumerate(results, start=1):
        print(f"\n[Result {idx}]")
        print(f"  Tool ID : {result['tool_use_id']}")
        print(f"  Type    : {result['type']}")
        print(f"  Content : {result['content']}")
    print("\n" + "="*42)


if __name__ == "__main__":
    asyncio.run(test_async_tool())

```

### Execution Invariants Achieved:

* **Task Futures Bundling:** By storing `asyncio.create_task()` pointers inside the `tasks` array, all three tools run instantly in parallel across separate worker threads via `asyncio.to_thread`.
* **Dynamic Interleaving Log Outputs:** When running this code, you will notice the `🔧 Tool called:` text prompts appear on the screen simultaneously, confirming the sequential execution path has been removed!

## Integrating Parallel Tool Execution

Great job practicing parallel tool execution in isolation! Now it's time to integrate that pattern into your agent's main workflow so that multiple tools can execute concurrently during actual conversations.

Currently, your agent's run() method processes tool calls one at a time in a sequential loop. When Claude requests multiple tools in a single turn, each tool waits for the previous one to complete before starting. This creates unnecessary delays, especially when tools could run simultaneously without interfering with each other.

Your task is to transform the sequential tool execution into parallel execution by updating the run() method in src/agent.py. You'll need to:

    Create an empty tasks list right after initializing tool_results to collect async tool tasks.
    Replace the direct await self.call_tool() call with asyncio.create_task() to schedule the tool without waiting, and append the task to your tasks list instead of appending to tool_results.
    After the loop completes, check if the tasks list has any items, then use asyncio.gather() to execute all tasks concurrently and extend tool_results with the returned results.

Remember that handoffs should still be handled immediately with await since they transfer control to another agent. By batching regular tool calls and executing them in parallel, you'll dramatically improve your agent's efficiency when handling multiple conversations or complex multi-tool requests!

```
# agent.py
import asyncio
from anthropic import AsyncAnthropic


class Agent:
    BASE_SYSTEM_PROMPT = (
        "You are an autonomous agent that can take multiple tool-calling steps when helpful. "
        "The user only sees your response when you stop using tools, not your tool usage or reasoning steps. "
        "When you provide your answer without calling tools, make it complete and standalone.\n"
        "Additional instructions:\n"
    )

    def __init__(
        self,
        name,
        system_prompt="You are a helpful assistant.",
        model="claude-sonnet-4-6",
        tools=None,
        tool_schemas=None,
        handoffs=None,
        max_turns=10
    ):
        self.client = AsyncAnthropic()
        self.name = name
        self.model = model
        self.system_prompt = self.BASE_SYSTEM_PROMPT + system_prompt
        self.max_turns = max_turns

        self.tools = {} if tools is None else dict(tools)
        self.tool_schemas = [] if tool_schemas is None else list(tool_schemas)
        self.handoffs = [] if handoffs is None else list(handoffs)
        
        self.handoff_schema = {
            "name": "handoff",
            "description": "Transfer control to another specialized agent. Use this when the user's request is better handled by a different agent.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": f"Name of the agent to handoff to. Available agents: {[agent.name for agent in self.handoffs]}"
                    },
                    "reason": {
                        "type": "string", 
                        "description": "Brief explanation of why this handoff is needed"
                    }
                },
                "required": ["name", "reason"]
            }
        }
        
    def _extract_text(self, content):
        return "".join(
            block.text for block in content
            if getattr(block, "type", None) == "text"
        )    

    def _build_request_args(self, messages):
        request_args = {
            "model": self.model,
            "system": self.system_prompt,
            "messages": messages,
            "max_tokens": 8000,
        }

        all_tools = []
        
        if self.tool_schemas:
            all_tools.extend(self.tool_schemas)
        if self.handoffs:
            all_tools.append(self.handoff_schema)
        if all_tools:
            request_args["tools"] = all_tools

        return request_args

    async def call_tool(self, tool_use):
        tool_name = tool_use.name
        tool_input = tool_use.input or {}
        tool_use_id = tool_use.id

        print(f"🔧 Tool called: {tool_name}({tool_input})")
        
        try:
            tool_fn = self.tools[tool_name]
        except KeyError:
            result = f"Error: Tool {tool_name} not found"
        except Exception as e:
            result = f"Error: {str(e)}"
        else:
            try:
                result = str(await asyncio.to_thread(tool_fn, **tool_input))
            except Exception as e:
                result = f"Error: {str(e)}"

        return {
            "type": "tool_result",
            "tool_use_id": tool_use_id,
            "content": result
        }
        
    async def call_handoff(self, tool_use, messages):
        agent_name = tool_use.input.get("name")
        reason = tool_use.input.get("reason", "No reason provided")
        
        print(f"🔄 Handoff to: {agent_name}")
        print(f"📝 Reason: {reason}")
        
        try:
            target_agent = next(agent for agent in self.handoffs if agent.name == agent_name)
            clean_messages = messages[:-1] if messages and messages[-1]["role"] == "assistant" else messages
            result = await target_agent.run(clean_messages)

            return True, result
        except StopIteration:
            return False, {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": f"Handoff failed: Agent '{agent_name}' not found. Available agents: {[agent.name for agent in self.handoffs]}"
            }
        except Exception as e:
            return False, {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": f"Handoff failed: Error during handoff to '{agent_name}': {str(e)}"
            }

    async def run(self, input_messages):
        messages = input_messages.copy()

        turn = 0
        while turn < self.max_turns:
            turn += 1

            response = await self.client.messages.create(
                **self._build_request_args(messages)
            )

            messages.append({"role": "assistant", "content": response.content})

            if response.stop_reason == "tool_use":
                tool_results = []
                
                # TODO: Create an empty list called tasks to collect async tool tasks
                
                
                for content_item in response.content:
                    if content_item.type == "tool_use":
                        if content_item.name == "handoff":
                            handoff_success, handoff_result = await self.call_handoff(content_item, messages)
                            if handoff_success:
                                return handoff_result
                            else:
                                tool_results.append(handoff_result)
                        else:
                            # TODO: Instead of awaiting call_tool directly, use asyncio.create_task() to schedule it and append the task to the tasks list
                            tool_result = await self.call_tool(content_item)
                            
                            tool_results.append(tool_result)

                # TODO: Check if tasks list is not empty, then use asyncio.gather() to execute all tasks concurrently and extend tool_results with the results


                messages.append({
                    "role": "user",
                    "content": tool_results
                })
        
            else:
                response_text = self._extract_text(response.content)

                return messages, response_text

        raise Exception("Max turns reached")


# main.py
import asyncio
import json
from agent import Agent
from functions import (
    sum_numbers,
    multiply_numbers,
    subtract_numbers,
    divide_numbers,
    power,
    square_root
)

# Load the schemas from JSON file
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Create a dictionary mapping tool names to functions
tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers,
    "subtract_numbers": subtract_numbers,
    "divide_numbers": divide_numbers,
    "power": power,
    "square_root": square_root
}

async def main():
    # Write multiple prompts that we'll process concurrently
    prompts = [
        "Solve this: (2 + 3) * (4*4)",
        "Find the roots of x^2 - 5x + 6 = 0",
    ]

    # Create a single agent that will handle multiple conversations concurrently
    agent = Agent(
        name="math_assistant",
        system_prompt="You are a helpful math assistant.",
        tools=tools,
        tool_schemas=tool_schemas,
        max_turns=15
    )

    # Create a list of coroutine tasks, one for each prompt (not started yet)
    tasks = [
        agent.run([{"role": "user", "content": prompt}])
        for prompt in prompts
    ]

    # Run all tasks concurrently and wait for all to complete
    results = await asyncio.gather(*tasks)

    # Display the final response from each conversation
    for idx, (_, result) in enumerate(results, start=1):
        print(f"\n=== run {idx} ===")
        print(result)

# Entry point - creates event loop and runs the async main() function
if __name__ == "__main__":
    asyncio.run(main())

```

To fully unlock concurrent execution within a single turn, we need to modify how the agent handles tool calls in the loop.

Instead of freezing the turn cycle to execute each tool one by one, we will use `asyncio.create_task()` to wrap `self.call_tool()` calls into an array of active tasks. Once the loop finishes scanning Claude's response payload, we fire the entire list concurrently using `asyncio.gather()`.

Here is the fully integrated, non-blocking `agent.py` core file ready for parallel orchestration, followed by your test runtime script:

### 1. Fully Concurrent Agent Workflow (`agent.py`)

```python
# agent.py
import asyncio
from anthropic import AsyncAnthropic


class Agent:
    BASE_SYSTEM_PROMPT = (
        "You are an autonomous agent that can take multiple tool-calling steps when helpful. "
        "The user only sees your response when you stop using tools, not your tool usage or reasoning steps. "
        "When you provide your answer without calling tools, make it complete and standalone.\n"
        "Additional instructions:\n"
    )

    def __init__(
        self,
        name,
        system_prompt="You are a helpful assistant.",
        model="claude-sonnet-4-6",
        tools=None,
        tool_schemas=None,
        handoffs=None,
        max_turns=10
    ):
        self.client = AsyncAnthropic()
        self.name = name
        self.model = model
        self.system_prompt = self.BASE_SYSTEM_PROMPT + system_prompt
        self.max_turns = max_turns

        self.tools = {} if tools is None else dict(tools)
        self.tool_schemas = [] if tool_schemas is None else list(tool_schemas)
        self.handoffs = [] if handoffs is None else list(handoffs)
        
        self.handoff_schema = {
            "name": "handoff",
            "description": "Transfer control to another specialized agent. Use this when the user's request is better handled by a different agent.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": f"Name of the agent to handoff to. Available agents: {[agent.name for agent in self.handoffs]}"
                    },
                    "reason": {
                        "type": "string", 
                        "description": "Brief explanation of why this handoff is needed"
                    }
                },
                "required": ["name", "reason"]
            }
        }
        
    def _extract_text(self, content):
        return "".join(
            block.text for block in content
            if getattr(block, "type", None) == "text"
        )    

    def _build_request_args(self, messages):
        request_args = {
            "model": self.model,
            "system": self.system_prompt,
            "messages": messages,
            "max_tokens": 8000,
        }

        all_tools = []
        
        if self.tool_schemas:
            all_tools.extend(self.tool_schemas)
        if self.handoffs:
            all_tools.append(self.handoff_schema)
        if all_tools:
            request_args["tools"] = all_tools

        return request_args

    async def call_tool(self, tool_use):
        tool_name = tool_use.name
        tool_input = tool_use.input or {}
        tool_use_id = tool_use.id

        print(f"🔧 Tool called: {tool_name}({tool_input})")
        
        try:
            tool_fn = self.tools[tool_name]
        except KeyError:
            result = f"Error: Tool {tool_name} not found"
        except Exception as e:
            result = f"Error: {str(e)}"
        else:
            try:
                result = str(await asyncio.to_thread(tool_fn, **tool_input))
            except Exception as e:
                result = f"Error: {str(e)}"

        return {
            "type": "tool_result",
            "tool_use_id": tool_use_id,
            "content": result
        }
        
    async def call_handoff(self, tool_use, messages):
        agent_name = tool_use.input.get("name")
        reason = tool_use.input.get("reason", "No reason provided")
        
        print(f"🔄 Handoff to: {agent_name}")
        print(f"📝 Reason: {reason}")
        
        try:
            target_agent = next(agent for agent in self.handoffs if agent.name == agent_name)
            clean_messages = messages[:-1] if messages and messages[-1]["role"] == "assistant" else messages
            result = await target_agent.run(clean_messages)

            return True, result
        except StopIteration:
            return False, {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": f"Handoff failed: Agent '{agent_name}' not found. Available agents: {[agent.name for agent in self.handoffs]}"
            }
        except Exception as e:
            return False, {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": f"Handoff failed: Error during handoff to '{agent_name}': {str(e)}"
            }

    async def run(self, input_messages):
        messages = input_messages.copy()

        turn = 0
        while turn < self.max_turns:
            turn += 1

            response = await self.client.messages.create(
                **self._build_request_args(messages)
            )

            messages.append({"role": "assistant", "content": response.content})

            if response.stop_reason == "tool_use":
                tool_results = []
                
                # Create an empty list called tasks to collect async tool tasks
                tasks = []
                
                for content_item in response.content:
                    if content_item.type == "tool_use":
                        if content_item.name == "handoff":
                            handoff_success, handoff_result = await self.call_handoff(content_item, messages)
                            if handoff_success:
                                return handoff_result
                            else:
                                tool_results.append(handoff_result)
                        else:
                            # Instead of awaiting call_tool directly, use asyncio.create_task() 
                            # to schedule it and append the task to the tasks list
                            task = asyncio.create_task(self.call_tool(content_item))
                            tasks.append(task)

                # Check if tasks list is not empty, then use asyncio.gather() 
                # to execute all tasks concurrently and extend tool_results with the results
                if tasks:
                    resolved_results = await asyncio.gather(*tasks)
                    tool_results.extend(resolved_results)

                messages.append({
                    "role": "user",
                    "content": tool_results
                })
        
            else:
                response_text = self._extract_text(response.content)

                return messages, response_text

        raise Exception("Max turns reached")

```

---

### 2. Multi-Conversation Verification Engine (`main.py`)

```python
# main.py
import asyncio
import json
from agent import Agent
from functions import (
    sum_numbers,
    multiply_numbers,
    subtract_numbers,
    divide_numbers,
    power,
    square_root
)

# Load the schemas from JSON file
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Create a dictionary mapping tool names to functions
tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers,
    "subtract_numbers": subtract_numbers,
    "divide_numbers": divide_numbers,
    "power": power,
    "square_root": square_root
}

async def main():
    # Write multiple prompts that we'll process concurrently
    prompts = [
        "Solve this: (2 + 3) * (4*4)",
        "Find the roots of x^2 - 5x + 6 = 0",
    ]

    # Create a single agent that will handle multiple conversations concurrently
    agent = Agent(
        name="math_assistant",
        system_prompt="You are a helpful math assistant.",
        tools=tools,
        tool_schemas=tool_schemas,
        max_turns=15
    )

    # Create a list of coroutine tasks, one for each prompt (not started yet)
    tasks = [
        agent.run([{"role": "user", "content": prompt}])
        for prompt in prompts
    ]

    # Run all tasks concurrently and wait for all to complete
    results = await asyncio.gather(*tasks)

    # Display the final response from each conversation
    for idx, (_, result) in enumerate(results, start=1):
        print(f"\n=== run {idx} ===")
        print(result)

# Entry point - creates event loop and runs the async main() function
if __name__ == "__main__":
    asyncio.run(main())

```

### Key Performance Benefits In This Architecture:

1. **Parallel Multi-Tool Calls:** When Claude decides to perform complex calculations within a single block—like processing two separate pieces of a math equation at the same time—the tools now execute in parallel.
2. **Maximum Throughput:** Combined with the multi-prompt `asyncio.gather(*tasks)` in `main.py`, tool calls from *different* turns and *different* problems can interleave on the event loop seamlessly. This ensures your application utilizes maximum execution efficiency with zero blocking overhead!


## Creating a Synchronous Agent Wrapper

While asynchronous programming provides excellent performance benefits, there are times when you need to integrate your agent into existing synchronous codebases that don't use async / await patterns.

Your task is to create a synchronous wrapper method that bridges the gap between synchronous code and your asynchronous agent system. In src/agent.py, you'll add a run_sync() method that:

    Takes the same input_messages parameter as the asynchronous run() method
    Uses asyncio.run() to execute the asynchronous run() method internally
    Returns the result directly without requiring await

Then, in src/main.py, you'll demonstrate synchronous usage by looping through the prompts list and calling agent.run_sync() for each prompt, printing the results with their run numbers.

The beauty of this approach is that even though you're calling the agent synchronously, it still benefits from all the internal asynchronous optimizations, such as parallel tool execution. This wrapper makes your agent system flexible enough to work in any Python codebase, whether it uses asynchronous patterns or not!


```
# agent.py
import asyncio
from anthropic import AsyncAnthropic


class Agent:
    BASE_SYSTEM_PROMPT = (
        "You are an autonomous agent that can take multiple tool-calling steps when helpful. "
        "The user only sees your response when you stop using tools, not your tool usage or reasoning steps. "
        "When you provide your answer without calling tools, make it complete and standalone.\n"
        "Additional instructions:\n"
    )

    def __init__(
        self,
        name,
        system_prompt="You are a helpful assistant.",
        model="claude-sonnet-4-6",
        tools=None,
        tool_schemas=None,
        handoffs=None,
        max_turns=10
    ):
        self.client = AsyncAnthropic()
        self.name = name
        self.model = model
        self.system_prompt = self.BASE_SYSTEM_PROMPT + system_prompt
        self.max_turns = max_turns

        self.tools = {} if tools is None else dict(tools)
        self.tool_schemas = [] if tool_schemas is None else list(tool_schemas)
        self.handoffs = [] if handoffs is None else list(handoffs)
        
        self.handoff_schema = {
            "name": "handoff",
            "description": "Transfer control to another specialized agent. Use this when the user's request is better handled by a different agent.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": f"Name of the agent to handoff to. Available agents: {[agent.name for agent in self.handoffs]}"
                    },
                    "reason": {
                        "type": "string", 
                        "description": "Brief explanation of why this handoff is needed"
                    }
                },
                "required": ["name", "reason"]
            }
        }
        
    def _extract_text(self, content):
        return "".join(
            block.text for block in content
            if getattr(block, "type", None) == "text"
        )    

    def _build_request_args(self, messages):
        request_args = {
            "model": self.model,
            "system": self.system_prompt,
            "messages": messages,
            "max_tokens": 8000,
        }

        all_tools = []
        
        if self.tool_schemas:
            all_tools.extend(self.tool_schemas)
        if self.handoffs:
            all_tools.append(self.handoff_schema)
        if all_tools:
            request_args["tools"] = all_tools

        return request_args

    async def call_tool(self, tool_use):
        tool_name = tool_use.name
        tool_input = tool_use.input or {}
        tool_use_id = tool_use.id

        print(f"🔧 Tool called: {tool_name}({tool_input})")
        
        try:
            tool_fn = self.tools[tool_name]
        except KeyError:
            result = f"Error: Tool {tool_name} not found"
        except Exception as e:
            result = f"Error: {str(e)}"
        else:
            try:
                result = str(await asyncio.to_thread(tool_fn, **tool_input))
            except Exception as e:
                result = f"Error: {str(e)}"

        return {
            "type": "tool_result",
            "tool_use_id": tool_use_id,
            "content": result
        }
        
    async def call_handoff(self, tool_use, messages):
        agent_name = tool_use.input.get("name")
        reason = tool_use.input.get("reason", "No reason provided")
        
        print(f"🔄 Handoff to: {agent_name}")
        print(f"📝 Reason: {reason}")
        
        try:
            target_agent = next(agent for agent in self.handoffs if agent.name == agent_name)
            clean_messages = messages[:-1] if messages and messages[-1]["role"] == "assistant" else messages
            result = await target_agent.run(clean_messages)

            return True, result
        except StopIteration:
            return False, {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": f"Handoff failed: Agent '{agent_name}' not found. Available agents: {[agent.name for agent in self.handoffs]}"
            }
        except Exception as e:
            return False, {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": f"Handoff failed: Error during handoff to '{agent_name}': {str(e)}"
            }

    async def run(self, input_messages):
        messages = input_messages.copy()

        turn = 0
        while turn < self.max_turns:
            turn += 1

            response = await self.client.messages.create(
                **self._build_request_args(messages)
            )

            messages.append({"role": "assistant", "content": response.content})

            if response.stop_reason == "tool_use":
                tool_results = []
                tasks = []
                for content_item in response.content:
                    if content_item.type == "tool_use":
                        if content_item.name == "handoff":
                            handoff_success, handoff_result = await self.call_handoff(content_item, messages)
                            if handoff_success:
                                return handoff_result
                            else:
                                tool_results.append(handoff_result)
                        else:
                            tasks.append(asyncio.create_task(self.call_tool(content_item)))

                if tasks:
                    tool_results.extend(await asyncio.gather(*tasks))

                messages.append({
                    "role": "user",
                    "content": tool_results
                })
        
            else:
                response_text = self._extract_text(response.content)

                return messages, response_text

        raise Exception("Max turns reached")

    # TODO: Add a run_sync method that wraps the async run() method for synchronous usage
    # Hint: This method should use asyncio.run() to execute the async run() method and return its result


# main.py
import json
from agent import Agent
from functions import (
    sum_numbers,
    multiply_numbers,
    subtract_numbers,
    divide_numbers,
    power,
    square_root
)

# Load the schemas from JSON file
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Create a dictionary mapping tool names to functions
tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers,
    "subtract_numbers": subtract_numbers,
    "divide_numbers": divide_numbers,
    "power": power,
    "square_root": square_root
}

# Write multiple prompts that we'll process synchronously
prompts = [
    "Solve this: (2 + 3) * (4*4)",
    "Find the roots of x^2 - 5x + 6 = 0",
]

# Create a single agent that will handle conversations
agent = Agent(
    name="math_assistant",
    system_prompt="You are a helpful math assistant.",
    tools=tools,
    tool_schemas=tool_schemas,
    max_turns=15
)

# TODO: Loop through each prompt in the prompts list using enumerate with start=1

    # TODO: Call agent.run_sync() with the proper message format for each prompt (no await needed!)

    # TODO: Print the result with run number and final response text

```

Here is the complete, production-ready source code for your asynchronous agent framework with parallel tool execution capabilities, including a dedicated synchronous wrapper layer.

The system is organized into a clean, decoupled architecture: `functions.py` houses your core algorithmic tools, `schemas.json` declares your technical manifests, `agent.py` drives the non-blocking execution runtime loops, and `main.py` orchestrates the parent lifecycles.

---

### 3. The Asynchronous Agent Class Engine (`agent.py`)

This houses your core state runtime. It applies `AsyncAnthropic`, handles defensive thread pooling offloads via `asyncio.to_thread()`, coordinates asynchronous multi-agent handoffs, and runs non-blocking fan-out passes via `asyncio.gather()`.

```python
# agent.py
import asyncio
from anthropic import AsyncAnthropic


class Agent:
    BASE_SYSTEM_PROMPT = (
        "You are an autonomous agent that can take multiple tool-calling steps when helpful. "
        "The user only sees your response when you stop using tools, not your tool usage or reasoning steps. "
        "When you provide your answer without calling tools, make it complete and standalone.\n"
        "Additional instructions:\n"
    )

    def __init__(
        self,
        name,
        system_prompt="You are a helpful assistant.",
        model="claude-sonnet-4-6",
        tools=None,
        tool_schemas=None,
        handoffs=None,
        max_turns=10
    ):
        self.client = AsyncAnthropic()
        self.name = name
        self.model = model
        self.system_prompt = self.BASE_SYSTEM_PROMPT + system_prompt
        self.max_turns = max_turns

        # Protect against external mutable reference mutations
        self.tools = {} if tools is None else dict(tools)
        self.tool_schemas = [] if tool_schemas is None else list(tool_schemas)
        self.handoffs = [] if handoffs is None else list(handoffs)
        
        # Configure internal dynamic multi-agent handoff schema
        self.handoff_schema = {
            "name": "handoff",
            "description": "Transfer control to another specialized agent. Use this when the user's request is better handled by a different agent.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": f"Name of the agent to handoff to. Available agents: {[agent.name for agent in self.handoffs]}"
                    },
                    "reason": {
                        "type": "string", 
                        "description": "Brief explanation of why this handoff is needed"
                    }
                },
                "required": ["name", "reason"]
            }
        }
        
    def _extract_text(self, content):
        return "".join(
            block.text for block in content
            if getattr(block, "type", None) == "text"
        )    

    def _build_request_args(self, messages):
        request_args = {
            "model": self.model,
            "system": self.system_prompt,
            "messages": messages,
            "max_tokens": 8000,
        }
        all_tools = []
        if self.tool_schemas:
            all_tools.extend(self.tool_schemas)
        if self.handoffs:
            all_tools.append(self.handoff_schema)
        if all_tools:
            request_args["tools"] = all_tools

        return request_args

    async def call_tool(self, tool_use):
        """Asynchronously dispatches a synchronous tool function into an isolated worker thread."""
        tool_name = tool_use.name
        tool_input = tool_use.input or {}
        tool_use_id = tool_use.id

        print(f"🔧 [Agent Context: {self.name}] Tool called: {tool_name}({tool_input})")
        
        try:
            tool_fn = self.tools[tool_name]
        except KeyError:
            result = f"Error: Tool {tool_name} not found"
        except Exception as e:
            result = f"Error: {str(e)}"
        else:
            try:
                # Wrap blocking execution into a non-blocking background thread pool worker
                result = str(await asyncio.to_thread(tool_fn, **tool_input))
            except Exception as e:
                result = f"Error: {str(e)}"

        return {
            "type": "tool_result",
            "tool_use_id": tool_use_id,
            "content": result
        }
        
    async def call_handoff(self, tool_use, messages):
        """Asynchronously routes the conversation context buffer to a down-stream specialist agent."""
        agent_name = tool_use.input.get("name")
        reason = tool_use.input.get("reason", "No reason provided")
        
        print(f"🔄 [{self.name}] Initiating Handoff to Specialist -> {agent_name}")
        print(f"📝 Rationale: {reason}")
        
        try:
            target_agent = next(agent for agent in self.handoffs if agent.name == agent_name)
            clean_messages = messages[:-1] if messages and messages[-1]["role"] == "assistant" else messages
            result = await target_agent.run(clean_messages)
            return True, result
        except StopIteration:
            return False, {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": f"Handoff failed: Agent '{agent_name}' not found. Available agents: {[agent.name for agent in self.handoffs]}"
            }
        except Exception as e:
            return False, {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": f"Handoff failed: Error during handoff to '{agent_name}': {str(e)}"
            }

    async def run(self, input_messages):
        """Core asynchronous state machine execution method loop."""
        messages = input_messages.copy()
        turn = 0
        
        while turn < self.max_turns:
            turn += 1

            response = await self.client.messages.create(
                **self._build_request_args(messages)
            )

            messages.append({"role": "assistant", "content": response.content})

            if response.stop_reason == "tool_use":
                tool_results = []
                tasks = [] # Collect tasks for parallel fan-out
                
                for content_item in response.content:
                    if content_item.type == "tool_use":
                        if content_item.name == "handoff":
                            handoff_success, handoff_result = await self.call_handoff(content_item, messages)
                            if handoff_success:
                                return handoff_result
                            else:
                                tool_results.append(handoff_result)
                        else:
                            # Schedule tool execution immediately on the background event loop
                            task = asyncio.create_task(self.call_tool(content_item))
                            tasks.append(task)

                # If multiple independent tool actions are collected, fire them in parallel
                if tasks:
                    resolved_blocks = await asyncio.gather(*tasks)
                    tool_results.extend(resolved_blocks)

                messages.append({
                    "role": "user",
                    "content": tool_results
                })
        
            else:
                response_text = self._extract_text(response.content)
                return messages, response_text

        raise Exception("Max turns reached")

    def run_sync(self, input_messages):
        """Synchronous wrapper abstraction bridging async assets to classic linear runtimes."""
        return asyncio.run(self.run(input_messages))

```

---

### 4. System Runtime Execution Orchestration Wrapper (`main.py`)

This entrypoint demonstrates concurrent multi-turn conversation handling using `asyncio.gather()`, as well as linear operation management using the `run_sync()` wrapper interface.

```python
# main.py
import asyncio
import json
from agent import Agent
from functions import (
    sum_numbers,
    multiply_numbers,
    subtract_numbers,
    divide_numbers,
    power,
    square_root
)

# Load structural manifests from your configuration resource file
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Initialize lookup mapping dictionary matching schema strings to function objects
tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers,
    "subtract_numbers": subtract_numbers,
    "divide_numbers": divide_numbers,
    "power": power,
    "square_root": square_root
}

# Define independent cross-domain query datasets
concurrent_prompts = [
    "Solve this mathematical formula step-by-step: (12 + 18) * (5 * 5)",
    "Find the roots of the quadratic equation: x^2 - 5x + 6 = 0",
]

synchronous_prompts = [
    "What is the square root of 144?",
    "Calculate 2 to the power of 8."
]

# Instantiate your single master worker agent instance
agent = Agent(
    name="math_assistant",
    system_prompt="You are an expert math assistant. Always use tools for calculations to ensure 100% accuracy.",
    tools=tools,
    tool_schemas=tool_schemas,
    max_turns=15
)

async def run_async_pipeline():
    print("=====================================================================")
    print("🚀 PIPELINE 1: DISPATCHING CONCURRENT CONVERSATIONS VIA ASYNC/AWAIT")
    print("=====================================================================\n")
    
    # Non-blocking generation of tasks inside memory
    tasks = [
        agent.run([{"role": "user", "content": prompt}])
        for prompt in concurrent_prompts
    ]
    
    # Interleave processing variables over the active event loop
    results = await asyncio.gather(*tasks)
    
    for idx, (_, response_text) in enumerate(results, start=1):
        print(f"\n📊 === CONCURRENT RESULT BLOCK {idx} ===")
        print(response_text)


def run_sync_pipeline():
    print("\n=====================================================================")
    print("💼 PIPELINE 2: DISPATCHING SEQUENTIAL CONVERSATIONS VIA RUN_SYNC()")
    print("=====================================================================\n")
    
    # Driving the async engine without using any 'async' or 'await' keywords in this scope
    for run_idx, prompt in enumerate(synchronous_prompts, start=1):
        print(f"Executing Linear Turn {run_idx}...")
        _, response_text = agent.run_sync([{"role": "user", "content": prompt}])
        
        print(f"=== SYNCHRONOUS RESPONSE BLOCK {run_idx} ===")
        print(response_text)
        print("-" * 40)


if __name__ == "__main__":
    # 1. Execute the native asynchronous pipeline concurrently
    asyncio.run(run_async_pipeline())
    
    # 2. Execute the alternative isolated synchronous interface pipeline
    run_sync_pipeline()

```